In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import seaborn as sns
from collections import Counter

In [ ]:
active_centromeres = pd.read_csv("/g/korbel/hain/centromere/samplesheets/centromeres_full.active.length.cluster.tsv", sep="\t")
active_centromeres = active_centromeres[~pd.isna(active_centromeres["ARCHITECTURE"])]
all_centromeres = pd.read_csv("/g/korbel/hain/centromere/samplesheets/centromeres_full.tsv", sep="\t")
all_centromeres = all_centromeres[all_centromeres["NUM_CONTIGS"] == 1]
all_centromeres = active_centromeres.copy()

In [ ]:
def parse_ascairn (sample:str):
    df = pd.read_csv(f"results/{sample}.cen_type_all.txt", sep="\t")
    
    num_miss = 0
    
    errors = []
    
    df_analysis_data = []
    
    for _, row in df.iterrows():
        
        chrom = row["Chr"]
        
        if chrom in ["chrX", "chrY"]:
            continue
        
        print(chrom)
        
        ### get ground truth
        # results from assembly
        sample_data = active_centromeres[(active_centromeres["CHROM"] == chrom) & (active_centromeres["SAMPLE"] == sample)]
        if sample_data.shape[0] != 2:
            print("Not complete ground truth available for sample:", sample, "chrom:", chrom)
            num_miss += 1
            continue
        
        print("-" * 50)
        print(f"Ground truth for sample: {sample}, chrom: {chrom}")
        print(f"\tHOR_LENGTH:{sample_data['HOR_LENGTH'].mean()}\n\tCLUSTER:{str(sorted(sample_data['ARCHITECTURE'].tolist()))}")
        print("-" * 50)
        
        ### results from ascairn        
        sample_h1 = row["Haplotype_1"].replace(".h1", "").replace(".h2", "")
        sample_h2 = row["Haplotype_2"].replace(".h1", "").replace(".h2", "")        
        tmp_h1 = all_centromeres[(all_centromeres["CHROM"] == chrom) & (all_centromeres["SAMPLE"] == sample_h1)]
        tmp_h2 = all_centromeres[(all_centromeres["CHROM"] == chrom) & (all_centromeres["SAMPLE"] == sample_h2)]
        
        ascairn_architectures = []
        ascairn_lengths = []
        
        found_haplotype = True
        
        for hap, hap_df in zip(["Haplotype_1", "Haplotype_2"], [tmp_h1, tmp_h2]):
            
            if not found_haplotype:
                break
            
            # subset to HGSVC results
            hap_df_work = hap_df[hap_df["PROJECT"] == "HGSVC"]
            
            asm_to_use = None
            if hap_df_work[hap_df_work["ASM"] == "VERKKO"].shape[0] == 2:
                asm_to_use = "VERKKO"
            elif hap_df_work[hap_df_work["ASM"] == "HIFI"].shape[0] == 2:
                asm_to_use = "HIFI"
            
            if asm_to_use is not None:
                pass#print(f"{hap}: Two HGSVC {asm_to_use} haplotypes available")
            else:
                print("No complete HGSVC assemblies present, skipping sample:", sample, "chrom:", chrom)
                num_miss += 1
                errors.append(f"No complete pair of HGSVC assemblies")
                found_haplotype = False
                break
            
            
            ### I know the sample and I know the assembler
            ### extract the information about the haplotype
            if row[hap].split(".")[-1] == "h1":
                asm_hp_to_use = "HP1"
            elif row[hap].split(".")[-1] == "h2":
                asm_hp_to_use = "HP2"
            else:
                asm_hp_to_use = None
            
            if asm_hp_to_use is None:
                print("Could not determine haplotype for", row[hap])
                num_miss += 1
                errors.append(f"Could not determine haplotype")
                found_haplotype = False
                break

            if hap_df[(hap_df["ASM"] == asm_to_use) & (hap_df["HP"] == asm_hp_to_use)].shape[0] == 0:
                print("No matching haplotype found for", row[hap])
                errors.append(f"No matching haplotype")
                num_miss += 1
                found_haplotype = False
                break
            
            hap_cluster = hap_df[(hap_df["ASM"] == asm_to_use) & (hap_df["HP"] == asm_hp_to_use)]["ARCHITECTURE"].values[0]
            hap_length = hap_df[(hap_df["ASM"] == asm_to_use) & (hap_df["HP"] == asm_hp_to_use)]["HOR_LENGTH"].values[0]
            ascairn_architectures.append(hap_cluster)
            ascairn_lengths.append(hap_length)
        
        if found_haplotype is False:
            continue
        
        if len(ascairn_architectures) == 2 and len(ascairn_lengths) == 2:
            print("-" * 50)
            print(f"ASCAIRN results for sample: {sample}, chrom: {chrom}")
            print(f"\tHOR_LENGTH:{np.mean(ascairn_lengths)}\n\tCLUSTER:{str(sorted(ascairn_architectures))}")
            print("-" * 50)
        else:
            print(f"Incomplete ascairn haplotypes for sample {sample} chrom {chrom}", ascairn_architectures, ascairn_lengths)
            num_miss += 1
            errors.append(f"Incomplete ascairn haplotypes")
            continue

        data = {
            "SAMPLE":sample,
            "CHROM":chrom,
            "ASM_HOR_LENGTH":sample_data['HOR_LENGTH'].mean(),
            "ASM_CLUSTER":str(sorted(sample_data['ARCHITECTURE'].tolist())),
            "ASCAIRN_HOR_LENGTH":np.mean(ascairn_lengths),
            "ASCAIRN_CLUSTER":str(sorted(ascairn_architectures)),
            "ASCAIRN_CONTIG_LENGTH":np.mean([row["Contig_len_1"], row["Contig_len_2"]]),
        }
        df_analysis_data.append(data)
    
    
    return pd.DataFrame(df_analysis_data), num_miss, errors

In [ ]:
samples = pd.read_csv("ascairn_samples.csv", sep=",")["SAMPLE_NAME"].tolist()

In [ ]:
dfs = []
errors = []
num_miss = 0
for sample in samples:
    df_sample, num_miss_sample, e = parse_ascairn(sample)
    dfs.append(df_sample)
    num_miss += num_miss_sample
    errors.extend(e)
df_compare = pd.concat(dfs, ignore_index=True)

In [ ]:
chromosomes = [f"chr{x}" for x in range(1, 23) if (f"chr{x}" in df_compare["CHROM"].unique().tolist())]

## Get ascairn metrics

In [ ]:
### helper function, extract TP, FP, FN from the comparison between truth (assembly) and predicted (ascairn) allelese
def calc_metrics(df:pd.DataFrame, truth_col:str, pred_col:str, chromosomes=chromosomes):
    
    tp = 0
    fp = 0
    fn = 0
    tn = 0  # Not defined for this label-pair comparison; retained for completeness.
    exact_pair_matches = 0

    for _, row in df[df["CHROM"].isin(chromosomes)].iterrows():
        truth_alleles = row[truth_col].replace("'", "").replace("[", "").replace("]", "").split(", ")
        pred_alleles = row[pred_col].replace("'", "").replace("[", "").replace("]", "").split(", ")

        truth_counts = Counter(truth_alleles)
        pred_counts = Counter(pred_alleles)
        matched = sum((truth_counts & pred_counts).values())

        tp += matched
        fp += len(pred_alleles) - matched
        fn += len(truth_alleles) - matched
        exact_pair_matches += truth_counts == pred_counts
        
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    accuracy = tp / (tp + fp + fn)
    exact_pair_accuracy = exact_pair_matches / len(df)

    return pd.DataFrame([{
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "PRECISION": precision,
        "RECALL": recall,
        "ACCURACY": accuracy,
        "EXACT_PAIR_ACCURACY": exact_pair_accuracy,
    }])

In [ ]:
test_sets = {}
for seed in [10, 21, 31]:
    test_sets[seed] = [x.strip() for x in open(f"../test_samples_1kGP_seed_{seed}.txt").readlines()]

In [ ]:
prec_recall_data = []

for seed in [10, 21, 31]:
    
    test_set = test_sets[seed]

    for chrom in chromosomes:
    
        ### get asm length
        y_asm = df_compare[df_compare["SAMPLE"].isin(test_set) & df_compare["CHROM"].eq(chrom)]["ASM_HOR_LENGTH"]
        y_ascairn = df_compare[df_compare["SAMPLE"].isin(test_set) & df_compare["CHROM"].eq(chrom)]["ASCAIRN_HOR_LENGTH"]
        r2_asm_ascairn = r2_score(y_asm, y_ascairn)
        if r2_asm_ascairn < 0:
            r2_asm_ascairn = 0

        prec_recall_data.append({
            "CHROM":chrom, 
            "Tool":"ASCAIRN", 
            "PRECISION":calc_metrics(df_compare[df_compare["SAMPLE"].isin(test_set)], truth_col = "ASM_CLUSTER", pred_col = "ASCAIRN_CLUSTER", chromosomes=[chrom])["PRECISION"].values[0],
            "RECALL":calc_metrics(df_compare[df_compare["SAMPLE"].isin(test_set)], truth_col = "ASM_CLUSTER", pred_col = "ASCAIRN_CLUSTER", chromosomes=[chrom])["RECALL"].values[0],
            "ACCURACY":calc_metrics(df_compare[df_compare["SAMPLE"].isin(test_set)], truth_col = "ASM_CLUSTER", pred_col = "ASCAIRN_CLUSTER", chromosomes=[chrom])["ACCURACY"].values[0],
            "R2": r2_asm_ascairn,
            "SEED":seed
        })
    
    
prec_recall_df = pd.DataFrame(prec_recall_data)
prec_recall_df_melt = pd.melt(
    prec_recall_df,
    id_vars=["CHROM", "Tool", "SEED"],
    value_vars=["PRECISION", "RECALL", "ACCURACY", "R2"],
    value_name="Value",
    var_name="Metric"
)
prec_recall_df_melt["Metric"] = prec_recall_df_melt["Metric"].replace({
    "PRECISION": "Precision",
    "RECALL": "Recall",
    "ACCURACY": "Accuracy",
    "R2": "R2"
})
prec_recall_df_melt["GROUP"] = "ASCAIRN"

In [ ]:
prec_recall_df_melt

In [ ]:
hor_df = pd.read_csv("../plotting/horoscope_metrics_train_test.tsv", sep="\t")
hor_df["GROUP"] = hor_df["GROUP"].replace({"TRAIN":"HOROSCOPE_TRAIN", "TEST":"HOROSCOPE_TEST"})

In [ ]:
chromosomes = hor_df["CHROM"].unique().tolist()

In [ ]:
plot_df = pd.concat([hor_df, prec_recall_df_melt]).reset_index(drop=True)

In [ ]:
plot_df = plot_df[plot_df["CHROM"].isin(chromosomes)]

In [ ]:
summary_plot_df = plot_df.copy()

In [ ]:
summary_plot_df["GROUP"] = summary_plot_df["GROUP"].replace({"FULL":"HOROSCOPE_FULL"})

In [ ]:
fig = plt.figure(figsize=(12, 16), layout="constrained")
gs = fig.add_gridspec(6, 2, width_ratios=[3, 1], height_ratios=[1, 1, 1, 1, 1, 0.1])

ax_top_left = fig.add_subplot(gs[0, 0])
ax_top_right = fig.add_subplot(gs[0, 1])

ax_top_left.set_ylim(0.90, 1.01)

axes_lower = [
    fig.add_subplot(gs[row, :])
    for row in range(1, 5)
]
legend_ax = fig.add_subplot(gs[5, :])
legend_ax.axis("off")

sns.boxplot(
    data=summary_plot_df,
    x="Metric",
    y="Value",
    hue="GROUP",
    hue_order=["HOROSCOPE_FULL", "HOROSCOPE_TRAIN", "HOROSCOPE_TEST", "ASCAIRN"],
    order=["Recall", "Precision", "Accuracy"],
    ax=ax_top_left
)

sns.boxplot(
    data=summary_plot_df,
    x="Metric",
    y="Value",
    hue="GROUP",
    hue_order=["HOROSCOPE_FULL", "HOROSCOPE_TRAIN", "HOROSCOPE_TEST", "ASCAIRN"],
    ax=ax_top_right,
    order=["R2"]
)

ax_top_left.set_title("Architecture prediction")
ax_top_right.set_title("HOR size prediction")

for plt_idx, metric in enumerate(["Accuracy", "Recall", "Precision", "R2"]):
    sns.barplot(
        data=summary_plot_df[summary_plot_df["Metric"] == metric], 
        x="CHROM", 
        y="Value", 
        hue="GROUP", 
        hue_order=["HOROSCOPE_FULL", "HOROSCOPE_TRAIN", "HOROSCOPE_TEST", "ASCAIRN"],
        ax=axes_lower[plt_idx], 
        order=[x for x in chromosomes if x in summary_plot_df["CHROM"].unique()],
        errorbar="sd"
    )
    axes_lower[plt_idx].set_title(metric)
    axes_lower[plt_idx].set_ylim(0, 1.05)

axes = [ax_top_left, *axes_lower]
handles, labels = ax_top_left.get_legend_handles_labels()
for ax in [*axes, ax_top_right]:
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()
legend_ax.legend(handles, labels, loc="center", ncol=4)

for label, ax in zip("ABCDE", axes):
    ax.text(
        -0.08, 1.05, label,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        va="bottom",
        ha="right",
    )

plt.savefig("../plotting/plots/ascairn_horoscope_metrics.train_test_ascairn.png")
plt.show()

In [ ]:
fig, (ax_left, ax_right) = plt.subplots(
    1, 2,
    figsize=(8, 4),
    gridspec_kw={"width_ratios": [3, 1]},
    layout="constrained",
)

sns.boxplot(
    data=summary_plot_df,
    x="Metric",
    y="Value",
    hue="GROUP",
    hue_order=["HOROSCOPE_FULL", "HOROSCOPE_TRAIN", "HOROSCOPE_TEST", "ASCAIRN"],
    order=["Recall", "Precision", "Accuracy"],
    ax=ax_left
)

sns.boxplot(
    data=summary_plot_df,
    x="Metric",
    y="Value",
    hue="GROUP",
    hue_order=["HOROSCOPE_FULL", "HOROSCOPE_TRAIN", "HOROSCOPE_TEST", "ASCAIRN"],
    ax=ax_right,
    order=["R2"]
)

ax_left.set_title("Architecture prediction")
ax_right.set_title("HOR size prediction")

ax_left.set_ylim(0.9, 1.01)

handles, labels = ax_left.get_legend_handles_labels()
ax_left.get_legend().remove()
ax_right.get_legend().remove()
fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.02),
    ncols=4,
)


plt.savefig("../plotting/plots/ascairn_horoscope_metrics.train_test_ascairn.summary.png", bbox_inches="tight")
plt.tight_layout()
plt.show()

In [ ]:
plot_df[
    (plot_df["GROUP"] == "ASCAIRN") &
    (plot_df["Metric"] == "Recall")
]["Value"].mean().round(2)

In [ ]:
plot_df[
    (plot_df["GROUP"] == "ASCAIRN") &
    (plot_df["Metric"] == "Precision") &
    (plot_df["Value"] > 0)
]["Value"].mean().round(2)

In [ ]:
plot_df[
    (plot_df["GROUP"] == "ASCAIRN") &
    (plot_df["Metric"] == "R2") &
    (plot_df["Value"] > 0)
]["Value"].mean().round(2)

In [ ]:
plot_df[
    (plot_df["GROUP"] == "HOROSCOPE_TEST") &
    (plot_df["Metric"] == "Recall") &
    (plot_df["Value"] > 0)
]["Value"].mean().round(2)

In [ ]:
plot_df[
    (plot_df["GROUP"] == "HOROSCOPE_TEST") &
    (plot_df["Metric"] == "Precision") &
    (plot_df["Value"] > 0)
]["Value"].mean().round(2)

In [ ]:
plot_df[
    (plot_df["GROUP"] == "HOROSCOPE_TEST") &
    (plot_df["Metric"] == "Recall") &
    (plot_df["Value"] > 0)
]["Value"].mean().round(2)

In [ ]:
plot_df[
    (plot_df["GROUP"] == "HOROSCOPE_TEST") &
    (plot_df["Metric"] == "R2") &
    (plot_df["Value"] > 0)
]["Value"].mean().round(2)